#### Load Data

In [252]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from itertools import product
from openpyxl import Workbook

# load dataset
df = pd.read_csv("../data/coffee_shop_sales.csv")

# display information
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Shape: (149116, 11)

Columns:
['Transaction ID', 'Transaction Date', 'Transaction Time', 'Transaction Quantity', 'Store ID', 'Store Location', 'Product ID', 'Unit Price', 'Product Category', 'Product Type', 'Product Detail']


#### Add Quarter Columns

In [253]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])

/var/folders/53/vr8gs__x60sg207tybh25bv80000gn/T/ipykernel_81889/3594840833.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])


In [254]:
df["Quarter"] = df["Transaction Date"].dt.to_period("Q")

print(df["Quarter"].value_counts().sort_index())

Quarter
2023Q1    54902
2023Q2    94214
Freq: Q-DEC, Name: count, dtype: int64


In [255]:
for quarter in df["Quarter"].sort_values().unique():
    quarter_df = df[df["Quarter"] == quarter]

    print(
        quarter,
        quarter_df["Transaction Date"].min(),
        quarter_df["Transaction Date"].max(),
        len(quarter_df)
    )

2023Q1 2023-01-01 00:00:00 2023-03-31 00:00:00 54902
2023Q2 2023-04-01 00:00:00 2023-06-30 00:00:00 94214


#### Calculate Business Metrics

In [256]:
quarterly_results = {}

quarters = df["Quarter"].sort_values().unique()

for quarter in quarters:

    # filter data per quarter
    quarter_df = df[df["Quarter"] == quarter].copy()

    # metrics
    quarter_df["Revenue"] = (
    quarter_df["Transaction Quantity"] *
    quarter_df["Unit Price"]
    )
    #------------------------------------------
    total_revenue = quarter_df["Revenue"].sum()

    total_transactions = quarter_df["Transaction ID"].nunique()

    total_units = quarter_df["Transaction Quantity"].sum()

    average_transaction_value = (
        total_revenue / total_transactions
    )

    average_units_per_transaction = (
        total_units / total_transactions
    )
    # print(f"\nMetrics for {quarter}:")
    # print("Revenue:", total_revenue)
    # print("Transactions:", total_transactions)
    # print("Units:", total_units)
    # print("Average Transaction Value:", average_transaction_value)
    # print("Average Units per Transaction:", average_units_per_transaction)


    # product related metrics

    product_sales = (
        quarter_df.groupby("Product Type")["Revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    category_sales = (
        quarter_df.groupby("Product Category")["Revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    quarter_df["Revenue %"] = (
        quarter_df["Revenue"] /
        quarter_df["Revenue"].sum()
    )


    # metrics by product

    product_sales_metrics = (
        quarter_df.groupby("Product Detail")
        .agg(
            revenue=("Revenue", "sum"),
            units_sold=("Transaction Quantity", "sum"),
            transactions=("Transaction ID", "nunique")
        )
        .sort_values("revenue", ascending=False)
    )


    # revenue divided into various time periods

    quarter_df["Transaction Date"] = pd.to_datetime(
        quarter_df["Transaction Date"],
        errors="coerce"
    )

    sales_by_day = (
        quarter_df.groupby("Transaction Date")["Revenue"]
        .sum()
    )

    quarter_df["Transaction Time"] = pd.to_datetime(
        quarter_df["Transaction Time"],
        format="%H:%M:%S",
        errors="coerce"
    )


    # extract hour

    quarter_df["Transaction Hour"] = (
        quarter_df["Transaction Time"].dt.hour
    )


    # sales by hour

    hourly_sales = (
        quarter_df.groupby("Transaction Hour")["Revenue"]
        .sum()
    )

    sales_by_hour = (
        quarter_df.groupby("Transaction Hour")
        .agg(
            revenue=("Revenue", "sum"),
            transactions=("Transaction ID", "nunique"),
            units_sold=("Transaction Quantity", "sum")
        )
        .sort_index()
    )

    peak_hour = hourly_sales.idxmax()


    # sales by day of week

    quarter_df["Day of Week"] = (
        quarter_df["Transaction Date"].dt.day_name()
    )

    day_order = [
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ]

    sales_by_weekday = (
        quarter_df.groupby("Day of Week")["Revenue"]
        .sum()
        .reindex(day_order)
    )


    # sales by time period

    def time_period(hour):
        if 6 <= hour < 10:
            return "Morning"
        elif 11 <= hour < 13:
            return "Lunch"
        elif 14 <= hour < 16:
            return "Afternoon"
        else:
            return "Evening"


    quarter_df["Time Period"] = (
        quarter_df["Transaction Hour"].apply(time_period)
    )

    time_order = [
        "Morning",
        "Lunch",
        "Afternoon",
        "Evening"
    ]

    quarter_df["Time Period"] = pd.Categorical(
        quarter_df["Time Period"],
        categories=time_order,
        ordered=True
    )

    sales_by_time_of_day = (
        quarter_df.groupby("Time Period")["Revenue"]
        .sum()
    )


    # revenue by product category and time period

    sales_by_period_and_category = (
        quarter_df.groupby(
            ["Time Period", "Product Category"]
        )["Revenue"]
        .sum()
        .unstack(fill_value=0)
    )


    # location related metrics

    location_sales = (
        quarter_df.groupby("Store Location")["Revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    store_sales = (
        quarter_df.groupby(["Store ID", "Store Location"])
        .agg(
            revenue=("Revenue", "sum"),
            units_sold=("Transaction Quantity", "sum"),
            transactions=("Transaction ID", "nunique")
        )
        .sort_values("revenue", ascending=False)
    )


    revenue_by_day_hour = {}

    for location in quarter_df["Store Location"].unique():

        revenue_by_day_hour[location] = (
            quarter_df[
                quarter_df["Store Location"] == location
            ]
            .groupby(
                ["Day of Week", "Transaction Hour"]
            )["Revenue"]
            .sum()
            .unstack(fill_value=0)
            .reindex(day_order)
        )


    # store all results for this quarter
    quarterly_results[quarter] = {

        "total_revenue": total_revenue,
        "total_transactions": total_transactions,
        "total_units": total_units,
        "average_transaction_value": average_transaction_value,
        "average_units_per_transaction": average_units_per_transaction,

        "product_sales": product_sales,
        "category_sales": category_sales,
        "product_detail_sales": product_sales_metrics,

        "sales_by_day": sales_by_day,
        "sales_by_hour": sales_by_hour,
        "sales_by_weekday": sales_by_weekday,
        "sales_by_time_of_day": sales_by_time_of_day,
        "sales_by_period_and_category": sales_by_period_and_category,

        "location_sales": location_sales,
        "store_sales": store_sales,
        "revenue_by_day_hour": revenue_by_day_hour
    }

In [257]:
# quarterly_results[pd.Period("2023Q1")]["category_sales"]

#### Automation

In [ ]:
from matplotlib.pyplot import bar
from openpyxl.styles import Font, Alignment
import datetime
from openpyxl.chart import BarChart, Reference, LineChart, DoughnutChart, ScatterChart, Reference, Series
from openpyxl.chart.shapes import GraphicalProperties
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.utils import get_column_letter

from openpyxl.chart.axis import ChartLines
from openpyxl.drawing.line import LineProperties
from openpyxl.styles.colors import Color

for quarter, results in quarterly_results.items():

    wb = Workbook()

    # formatting
    header_18 = Font(name='Calibri', size=18, bold=True, color="222222")
    header_14 = Font(name='Calibri', size=14, bold=True, color="222222")

    body_bold = Font(name='Calibri', size=11, bold=True, color="222222")
    body_regular = Font(name='Calibri', size=11, bold=False, color="222222")

    center_align = Alignment(horizontal='center', vertical='center')


    # ####### first sheet - KPIs #######
    kpi_sheet = wb.active

    # name it KPIs
    kpi_sheet.title = "KPIs"

    # set columns width
    kpi_sheet.column_dimensions['A'].width = 20
    kpi_sheet.column_dimensions['B'].width = 20
    kpi_sheet.column_dimensions['C'].width = 20
    kpi_sheet.column_dimensions['D'].width = 20
    kpi_sheet.column_dimensions['E'].width = 20

    # add todays date up top
    kpi_sheet['A1'] = datetime.datetime.now().strftime("%d %B, %Y")
    kpi_sheet.row_dimensions[1].height = 40

    # set header
    kpi_sheet['A2'] = "Sales Performance Report"
    kpi_sheet['A2'].font = Font(name='Calibri', size=28, bold=True, color="222222")
    kpi_sheet.row_dimensions[2].height = 60
    kpi_sheet.merge_cells('A2:E2')
    kpi_sheet['A2'].alignment = center_align

    # set subheader
    start_month = quarter.start_time.strftime("%B")
    end_month = quarter.end_time.strftime("%B")
    year = quarter.start_time.year
    #-----------------------------------
    kpi_sheet["A3"] = f"{start_month} – {end_month} {year}"
    kpi_sheet['A3'].font = header_18
    kpi_sheet.row_dimensions[3].height = 36
    kpi_sheet.merge_cells('A3:E3')
    kpi_sheet['A3'].alignment = center_align

    # blank row
    kpi_sheet["A4"] = ""
    kpi_sheet.row_dimensions[4].height = 30

    # FIRST ROW OF KPIs
    # headers
    kpi_sheet.row_dimensions[5].height = 20
    kpi_sheet["A5"] = "TOTAL REVENUE"
    kpi_sheet["C5"] = "TRANSACTIONS"
    kpi_sheet["E5"] = "AVG TRANSACTION"
    for cell in kpi_sheet[5]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[6].height = 20
    kpi_sheet["A6"] = f"${total_revenue:,.2f}"
    kpi_sheet["C6"] = total_transactions
    kpi_sheet["E6"] = f"${average_transaction_value:,.2f}"
    for cell in kpi_sheet[6]:
        cell.alignment = center_align
        cell.font = body_regular

    # SECOND ROW OF KPIs
    # blank row
    kpi_sheet["A7"] = ""
    kpi_sheet.row_dimensions[7].height = 30

    # headers
    kpi_sheet.row_dimensions[8].height = 20
    kpi_sheet["A8"] = "BEST LOCATION"
    kpi_sheet["C8"] = "BEST PRODUCT "
    kpi_sheet["E8"] = "PEAK PERIOD"
    for cell in kpi_sheet[8]:
        cell.alignment = center_align 
        cell.font = header_14

    # numbers
    kpi_sheet.row_dimensions[9].height = 20
    kpi_sheet["A9"] = store_sales.index[0][1]  # best location
    kpi_sheet["C9"] = product_sales.index[0]  # best product
    kpi_sheet["E9"] = f"{peak_hour}:00 - {peak_hour + 1}:00"  # peak period

    for cell in kpi_sheet[9]:
        cell.alignment = center_align
        cell.font = body_regular

    #################################################
    # END OF KPI SHEET
    #################################################

    # ######## SECOND SHEET - SALES ANALYSIS #######
    sales_analysis = wb.create_sheet(title="Sales Analysis")

    # set columns width
    sales_analysis.column_dimensions['A'].width = 18
    sales_analysis.column_dimensions['B'].width = 18
    sales_analysis.column_dimensions['C'].width = 18
    sales_analysis.column_dimensions['D'].width = 18
    sales_analysis.column_dimensions['E'].width = 18
    sales_analysis.column_dimensions['F'].width = 18
    sales_analysis.column_dimensions['G'].width = 18
    sales_analysis.column_dimensions['H'].width = 18
    sales_analysis.column_dimensions['I'].width = 18

    # set header
    sales_analysis['A1'] = "Sales Analysis"
    sales_analysis['A1'].font = header_18
    sales_analysis.row_dimensions[1].height = 60
    sales_analysis.merge_cells('A1:I1')
    sales_analysis['A1'].alignment = center_align

    # set subheader
    sales_analysis['A2'] = "Revenue performance across products and categories"
    sales_analysis['A2'].font = header_14
    sales_analysis.row_dimensions[2].height = 36
    sales_analysis.merge_cells('A2:I2')
    sales_analysis['A2'].alignment = center_align

    # blank row
    sales_analysis["A3"] = ""
    sales_analysis.row_dimensions[3].height = 30

    # CHART DATA SHEET
    chart_data_ws = wb.create_sheet("Chart Data")

    # REVENUE BY CATEGORY
    category_data = category_sales.reset_index()

    category_data.columns = ["Product Category", "Revenue"]

    chart_data_ws["A1"] = "Product Category"
    chart_data_ws["B1"] = "Revenue"

    for row_num, row in enumerate(
        category_data.itertuples(index=False),
        start=2
        ):
        chart_data_ws.cell(row=row_num, column=1, value=row[0])
        chart_data_ws.cell(row=row_num, column=2, value=row[1])

    sales_analysis['A4'] = "Revenue by Product Category"
    sales_analysis['A4'].font = header_14
    sales_analysis.row_dimensions[4].height = 36
    
    category_chart = BarChart()

    category_chart.type = "bar"
    category_chart.style = 13

    category_chart.legend = None

    category_chart.x_axis.majorGridlines = None
    category_chart.y_axis.majorGridlines = None

    category_chart.x_axis.delete = False
    category_chart.y_axis.delete = False

    category_chart.x_axis.majorTickMark = "out"
    category_chart.y_axis.majorTickMark = "out"

    category_chart.x_axis.tickLblPos = "nextTo"
    category_chart.y_axis.tickLblPos = "nextTo"

    category_chart.y_axis.majorUnit = 25000

    category_chart.x_axis.tickLblPos = "low"
    category_chart.x_axis.numFmt = '$#,##0'

    category_data_ref = Reference(
        chart_data_ws,
        min_col=2,
        min_row=1,
        max_row=len(category_data) + 1
    )

    category_labels = Reference(
        chart_data_ws,
        min_col=1,
        min_row=2,
        max_row=len(category_data) + 1
    )

    category_chart.add_data(
        category_data_ref,
        titles_from_data=True
    )

    category_chart.set_categories(category_labels)
    
    category_chart.height = 10
    category_chart.width = 15

    sales_analysis.row_dimensions[5].height = category_chart.height * 28.35

    sales_analysis.add_chart(
        category_chart,
        "A5"
    )

    # REVENUE BY MONTH
    monthly_sales = results["sales_by_day"].copy()

    monthly_sales.index = pd.to_datetime(monthly_sales.index)

    monthly_sales = (
        monthly_sales
        .resample("ME")
        .sum()
    )

    monthly_data = monthly_sales.reset_index()

    monthly_data.columns = ["Month", "Revenue"]

    chart_data_ws["D1"] = "Month"
    chart_data_ws["E1"] = "Revenue"

    for row_num, row in enumerate(
        monthly_data.itertuples(index=False),
        start=2
    ):
        chart_data_ws.cell(
            row=row_num,
            column=4,
            value=row[0].strftime("%B")
        )
        chart_data_ws.cell(
            row=row_num,
            column=5,
            value=row[1]
        )

    sales_analysis['F4'] = "Revenue by Month"
    sales_analysis['F4'].font = header_14
    sales_analysis.row_dimensions[4].height = 36

    monthly_chart = LineChart()
    monthly_chart.style = 13
    monthly_chart.legend = None

    monthly_data_ref = Reference(
        chart_data_ws,
        min_col=5,
        min_row=1,
        max_row=len(monthly_data) + 1
    )

    monthly_labels = Reference(
        chart_data_ws,
        min_col=4,
        min_row=2,
        max_row=len(monthly_data) + 1
    )

    monthly_chart.add_data(
        monthly_data_ref,
        titles_from_data=True
    )

    monthly_chart.set_categories(
    monthly_labels
    )

    monthly_chart.x_axis.majorTickMark = "out"
    monthly_chart.y_axis.majorTickMark = "out"

    monthly_chart.x_axis.majorGridlines = None
    monthly_chart.y_axis.majorGridlines = None

    monthly_chart.x_axis.delete = False
    monthly_chart.y_axis.delete = False

    monthly_chart.x_axis.tickLblPos = "low"
    monthly_chart.y_axis.tickLblPos = "nextTo"

    monthly_chart.y_axis.majorUnit = 25000
    monthly_chart.y_axis.numFmt = '$#,##0'

    monthly_chart.height = 10
    monthly_chart.width = 15

    sales_analysis.row_dimensions[5].height = monthly_chart.height * 28.35

    sales_analysis.add_chart(
        monthly_chart,
        "F5"
    )
    # blank row
    sales_analysis["A6"] = ""
    sales_analysis.row_dimensions[6].height = 30

    # REVENUE BY TIME OF DAY
    sales_analysis["A7"] = "Revenue by Time of Day"
    sales_analysis["A7"].font = header_14
    sales_analysis.row_dimensions[7].height = 36

    time_sales = results["sales_by_time_of_day"]

    # chart data
    chart_data_ws["G1"] = "Time Period"
    chart_data_ws["H1"] = "Revenue"

    for row_num, (time_period, revenue) in enumerate(
        time_sales.items(),
        start=2
    ):
        chart_data_ws.cell(
            row=row_num,
            column=7,
            value=time_period
        )

        chart_data_ws.cell(
            row=row_num,
            column=8,
            value=revenue
        )

    # create chart
    time_chart = DoughnutChart()

    time_data_ref = Reference(
        chart_data_ws,
        min_col=8,
        min_row=1,
        max_row=len(time_sales) + 1
    )

    time_labels = Reference(
        chart_data_ws,
        min_col=7,
        min_row=2,
        max_row=len(time_sales) + 1
    )

    time_chart.add_data(
        time_data_ref,
        titles_from_data=True
    )

    time_chart.set_categories(time_labels)
    time_chart.style = 13

    time_chart.height = 10
    time_chart.width = 15
    time_chart.holeSize = 45
    time_chart.legend.position = "r"

    sales_analysis.row_dimensions[8].height = time_chart.height * 28.35

    # add chart
    sales_analysis.add_chart(
        time_chart,
        "A8"
    )

    # TRANSACTIONS PER DAY VS REVENUE

    sales_analysis["F7"] = "Transactions per Day vs Revenue"
    sales_analysis["F7"].font = header_14
    sales_analysis.row_dimensions[7].height = 36

    # DAILY LOCATION DATA

    daily_location_sales = (
        quarter_df
        .groupby([
            "Store Location",
            "Transaction Date"
        ])
        .agg(
            revenue=("Revenue", "sum"),
            transactions=("Transaction ID", "nunique")
        )
        .reset_index()
    )

    # WRITE CHART DATA

    locations = [
        "Astoria",
        "Hell's Kitchen",
        "Lower Manhattan"
    ]

    chart_data_ws["M1"] = "Astoria Transactions"
    chart_data_ws["N1"] = "Astoria Revenue"

    chart_data_ws["P1"] = "Hell's Kitchen Transactions"
    chart_data_ws["Q1"] = "Hell's Kitchen Revenue"

    chart_data_ws["S1"] = "Lower Manhattan Transactions"
    chart_data_ws["T1"] = "Lower Manhattan Revenue"


    for col_start, location in zip(
        [13, 16, 19],
        locations
    ):

        location_data = (
            daily_location_sales[
                daily_location_sales["Store Location"] == location
            ]
            .sort_values("Transaction Date")
        )

        for row_num, row in enumerate(
            location_data.itertuples(index=False),
            start=2
        ):

            chart_data_ws.cell(
                row=row_num,
                column=col_start,
                value=row.transactions
            )

            chart_data_ws.cell(
                row=row_num,
                column=col_start + 1,
                value=row.revenue
            )

    # CREATE SCATTER CHART
    scatter_chart = ScatterChart()

    scatter_chart.style = 13

    # NO AXIS TITLES
    scatter_chart.x_axis.title = None
    scatter_chart.y_axis.title = None

    scatter_chart.x_axis.delete = False
    scatter_chart.y_axis.delete = False

    # X-AXIS RANGE

    min_transactions = daily_location_sales["transactions"].min()
    max_transactions = daily_location_sales["transactions"].max()

    scatter_chart.x_axis.scaling.min = min_transactions - 10
    scatter_chart.x_axis.scaling.max = max_transactions + 10

    # AXIS TICKS
    scatter_chart.x_axis.majorTickMark = "out"
    scatter_chart.y_axis.majorTickMark = "out"

    scatter_chart.x_axis.tickLblPos = "low"
    scatter_chart.y_axis.tickLblPos = "nextTo"

    # NO GRIDLINES
    scatter_chart.x_axis.majorGridlines = None
    scatter_chart.y_axis.majorGridlines = None

    # REVENUE FORMAT
    scatter_chart.y_axis.numFmt = '$#,##0'

    scatter_chart.height = 10
    scatter_chart.width = 15
    sales_analysis.row_dimensions[8].height = scatter_chart.height * 28.35

    # ADD EACH LOCATION AS A SEPARATE SERIES

    location_columns = [
        (13, 14, "Astoria"),
        (16, 17, "Hell's Kitchen"),
        (19, 20, "Lower Manhattan")
    ]

    for x_col, y_col, location in location_columns:

        location_data = (
            daily_location_sales[
                daily_location_sales["Store Location"] == location
            ]
        )

        x_values = Reference(
            chart_data_ws,
            min_col=x_col,
            min_row=2,
            max_row=len(location_data) + 1
        )

        y_values = Reference(
            chart_data_ws,
            min_col=y_col,
            min_row=2,
            max_row=len(location_data) + 1
        )

        series = Series(
            y_values,
            x_values,
            title=location
        )

        # DOTS ONLY

        series.marker.symbol = "circle"
        series.marker.graphicalProperties.shadow = None

        # NO CONNECTING LINES

        series.graphicalProperties.line.noFill = True

        scatter_chart.series.append(series)

    # LEGEND
    scatter_chart.legend.position = "r"

    # ADD CHART

    sales_analysis.add_chart(
        scatter_chart,
        "F8"
    )

    # blank row
    sales_analysis["A9"] = ""
    sales_analysis.row_dimensions[9].height = 30

    # Multi-Level Column Chart — Grouped by Hour, then by Day
    sales_analysis['A10'] = "Comparative Location Performance Grouped by Hour"
    sales_analysis['A10'].font = header_14
    sales_analysis.row_dimensions[10].height = 36
    sales_analysis.merge_cells('A10:I10')
    sales_analysis['A10'].alignment = center_align
    heatmap_start_row = 10

    locations = ["Astoria", "Hell's Kitchen", "Lower Manhattan"]
    day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    day_labels = {
        "Monday": "M",
        "Tuesday": "T",
        "Wednesday": "W",
        "Thursday": "T",
        "Friday": "F",
        "Saturday": "S",
        "Sunday": "S"
    }

    # Set step to 1 to include every single hour from 8 to 20
    hour_step = 1  
    heatmap_hours = list(range(8, 21, hour_step))

    data_start_col = 22

    # Write headers for data alignment (Hour first, Day is second)
    chart_data_ws.cell(row=2, column=data_start_col, value="Hour")
    chart_data_ws.cell(row=2, column=data_start_col + 1, value="Day")

    for loc_idx, location in enumerate(locations):
        chart_data_ws.cell(row=2, column=data_start_col + 2 + loc_idx, value=location)

    # Populate the text grid row by row
    current_row = 3
    for hour in heatmap_hours:
        is_first_day_of_hour = True
        
        for day in day_order:
            # Excel Multi-level rule: Only write the hour label on the first row of the hour block
            if is_first_day_of_hour:
                chart_data_ws.cell(row=current_row, column=data_start_col, value=f"{hour}:00")
                is_first_day_of_hour = False
            else:
                chart_data_ws.cell(row=current_row, column=data_start_col, value="")
                
            # The secondary category level displays the weekday underneath
            chart_data_ws.cell(row=current_row, column=data_start_col + 1, value=day_labels[day])
            
            # Populate metric value variables across location columns
            for loc_idx, location in enumerate(locations):
                heatmap_data = results["revenue_by_day_hour"][location]
                try:
                    val = float(heatmap_data.loc[day, hour])
                except KeyError:
                    val = 0.0
                chart_data_ws.cell(row=current_row, column=data_start_col + 2 + loc_idx, value=val)
                
            current_row += 1

    # Generate Clustered 2D Column Chart
    location_hour_chart = BarChart()
    location_hour_chart.type = "col"
    location_hour_chart.style = 10

    # Setting an ultra-wide canvas so all distinct data intervals fit perfectly
    location_hour_chart.height = 9
    location_hour_chart.width = 34  
    sales_analysis.row_dimensions[11].height = location_hour_chart.height * 28.35

    # Legend position top right corner
    location_hour_chart.legend.position = "tr"

    # Force multi-level labels to show up cleanly on the X-axis
    location_hour_chart.x_axis.noMultiLvlLbl = False
    location_hour_chart.x_axis.delete = False
    location_hour_chart.y_axis.delete = False

    # Create custom horizontal gridlines using a light shade of gray
    location_hour_chart.y_axis.majorGridlines = ChartLines()
    grid_line_properties = LineProperties()
    grid_line_properties.solidFill = "D3D3D3"
    # location_hour_chart.y_axis.majorGridlines.spPr.ln = grid_line_properties

    # References targeting numerical metric columns
    data_ref = Reference(
        chart_data_ws, 
        min_col=data_start_col + 2, 
        max_col=data_start_col + 1 + len(locations), 
        min_row=2, 
        max_row=current_row - 1
    )

    # References targeting text category index ranges
    categories_ref = Reference(
        chart_data_ws, 
        min_col=data_start_col, 
        max_col=data_start_col + 1, 
        min_row=3, 
        max_row=current_row - 1
    )

    location_hour_chart.add_data(
        data_ref,
        titles_from_data=True
    )

    location_hour_chart.set_categories(
        categories_ref
    )

    # Mount layout back onto dashboard page area
    sales_analysis.add_chart(
        location_hour_chart,
        "A11"
    )

    # Hide the chart data sheet
    chart_data_ws.sheet_state = "hidden"

    #################################################
    # END OF ANALYSIS SHEET
    #################################################
    
    
    
    ######## THIRD SHEET - OBSERVATIONS #######

def calculate_percent_change(current, previous):

    if previous == 0:
        return None

    return ((current - previous) / previous) * 100


def add_observation(
    observations,
    observation,
    metric,
    finding,
    investigation
):

    observations.append({
        "Observation": observation,
        "Metric": metric,
        "Finding": finding,
        "Suggested Investigation": investigation
    })


quarter_list = sorted(quarterly_results.keys())

quarterly_observations = {}

for quarter_index, quarter in enumerate(quarter_list):

    results = quarterly_results[quarter]

    observations = []

    # Current quarter metrics

    total_revenue = results["total_revenue"]
    total_transactions = results["total_transactions"]
    total_units = results["total_units"]
    average_transaction_value = results["average_transaction_value"]

    # Compare with previous quarter when available

    if quarter_index > 0:

        previous_quarter = quarter_list[quarter_index - 1]
        previous_results = quarterly_results[previous_quarter]

        previous_revenue = previous_results["total_revenue"]
        previous_transactions = previous_results["total_transactions"]
        previous_avg_transaction = (
            previous_results["average_transaction_value"]
        )

        revenue_change = calculate_percent_change(
            total_revenue,
            previous_revenue
        )

        transaction_change = calculate_percent_change(
            total_transactions,
            previous_transactions
        )

        avg_transaction_change = calculate_percent_change(
            average_transaction_value,
            previous_avg_transaction
        )

        # Revenue change

        if revenue_change is not None and abs(revenue_change) >= 10:

            direction = (
                "increased"
                if revenue_change > 0
                else "decreased"
            )

            add_observation(
                observations,
                "Revenue change",
                f"{revenue_change:+.1f}%",
                f"Revenue {direction} by {abs(revenue_change):.1f}% "
                f"compared with the previous quarter.",
                "Identify the locations, categories, and products "
                "responsible for the change."
            )

        # Transaction change

        if transaction_change is not None and abs(transaction_change) >= 10:

            direction = (
                "increased"
                if transaction_change > 0
                else "decreased"
            )

            add_observation(
                observations,
                "Transaction change",
                f"{transaction_change:+.1f}%",
                f"Transaction volume {direction} by "
                f"{abs(transaction_change):.1f}%.",
                "Review whether the change is concentrated "
                "in specific locations, days, or time periods."
            )

        # Average transaction value

        if (
            avg_transaction_change is not None
            and abs(avg_transaction_change) >= 10
        ):

            direction = (
                "increased"
                if avg_transaction_change > 0
                else "decreased"
            )

            add_observation(
                observations,
                "Average transaction value",
                f"{avg_transaction_change:+.1f}%",
                f"Average transaction value {direction} by "
                f"{abs(avg_transaction_change):.1f}%.",
                "Review changes in product mix and the number "
                "of items purchased per transaction."
            )

        # Revenue vs transaction volume

        if (
            revenue_change is not None
            and transaction_change is not None
        ):

            if (
                transaction_change >= 10
                and revenue_change < transaction_change - 10
            ):

                add_observation(
                    observations,
                    "Revenue vs transactions",
                    f"Revenue {revenue_change:+.1f}% / "
                    f"Transactions {transaction_change:+.1f}%",
                    "Transactions increased faster than revenue.",
                    "Investigate whether customers are making "
                    "smaller purchases or shifting toward lower-priced products."
                )

            elif (
                revenue_change >= 10
                and transaction_change < revenue_change - 10
            ):

                add_observation(
                    observations,
                    "Revenue vs transactions",
                    f"Revenue {revenue_change:+.1f}% / "
                    f"Transactions {transaction_change:+.1f}%",
                    "Revenue increased faster than transaction volume.",
                    "Investigate whether higher-priced products "
                    "or larger purchases are driving the increase."
                )

    # Location concentration

    location_sales = results["location_sales"]

    location_share = (
        location_sales / total_revenue
    ) * 100

    for location, share in location_share.items():

        if share >= 40:

            add_observation(
                observations,
                "Location concentration",
                f"{share:.1f}%",
                f"{location} generated {share:.1f}% "
                f"of quarterly revenue.",
                "Review whether revenue concentration is driven "
                "by transaction volume, customer behavior, or product mix."
            )

    # Location performance gap

    if len(location_sales) >= 2:

        highest_location = location_sales.idxmax()
        lowest_location = location_sales.idxmin()

        highest_revenue = location_sales.max()
        lowest_revenue = location_sales.min()

        if lowest_revenue > 0:

            location_gap = (
                (highest_revenue - lowest_revenue)
                / lowest_revenue
            ) * 100

            if location_gap >= 25:

                add_observation(
                    observations,
                    "Location performance gap",
                    f"{location_gap:.1f}%",
                    f"{highest_location} generated "
                    f"{location_gap:.1f}% more revenue than "
                    f"{lowest_location}.",
                    "Compare transaction volume, average transaction "
                    "value, product mix, and time-of-day patterns."
                )

    # Category concentration

    category_sales = results["category_sales"]

    category_share = (
        category_sales / total_revenue
    ) * 100

    for category, share in category_share.items():

        if share >= 30:

            add_observation(
                observations,
                "Category concentration",
                f"{share:.1f}%",
                f"{category} accounted for {share:.1f}% "
                f"of quarterly revenue.",
                "Review whether revenue is overly dependent on "
                "this category and which products drive its performance."
            )

    # Product concentration

    product_detail_sales = results["product_detail_sales"]

    product_share = (
        product_detail_sales["revenue"] / total_revenue
    ) * 100

    for product, share in product_share.items():

        if share >= 15:

            add_observation(
                observations,
                "Product concentration",
                f"{share:.1f}%",
                f"{product} accounted for {share:.1f}% "
                f"of quarterly revenue.",
                "Review the product's performance across locations "
                "and time periods."
            )

    # Time-of-day concentration

    time_sales = results["sales_by_time_of_day"]

    time_share = (
        time_sales / total_revenue
    ) * 100

    for time_period, share in time_share.items():

        if share >= 35:

            add_observation(
                observations,
                "Time-of-day concentration",
                f"{share:.1f}%",
                f"{time_period} generated {share:.1f}% "
                f"of quarterly revenue.",
                "Review whether staffing, inventory, and operations "
                "are aligned with demand during this period."
            )

        elif share <= 15:

            add_observation(
                observations,
                "Low-demand time period",
                f"{share:.1f}%",
                f"{time_period} generated only {share:.1f}% "
                f"of quarterly revenue.",
                "Investigate whether low demand is consistent "
                "across locations and whether the period warrants attention."
            )

    # Peak hour

    sales_by_hour = results["sales_by_hour"]

    peak_hour = sales_by_hour["revenue"].idxmax()
    peak_hour_revenue = sales_by_hour["revenue"].max()
    average_hour_revenue = sales_by_hour["revenue"].mean()

    if average_hour_revenue > 0:

        peak_hour_difference = (
            (peak_hour_revenue - average_hour_revenue)
            / average_hour_revenue
        ) * 100

        if peak_hour_difference >= 25:

            add_observation(
                observations,
                "Peak hour",
                f"{peak_hour_difference:.1f}% above average",
                f"{peak_hour}:00 generated "
                f"{peak_hour_difference:.1f}% more revenue "
                f"than the average hour.",
                "Review staffing and inventory requirements "
                "around the peak hour."
            )

    # Day-of-week performance

    sales_by_weekday = results["sales_by_weekday"]

    average_daily_revenue = sales_by_weekday.mean()

    highest_day = sales_by_weekday.idxmax()
    lowest_day = sales_by_weekday.idxmin()

    highest_day_revenue = sales_by_weekday.max()
    lowest_day_revenue = sales_by_weekday.min()

    if average_daily_revenue > 0:

        highest_day_difference = (
            (highest_day_revenue - average_daily_revenue)
            / average_daily_revenue
        ) * 100

        lowest_day_difference = (
            (lowest_day_revenue - average_daily_revenue)
            / average_daily_revenue
        ) * 100

        if highest_day_difference >= 20:

            add_observation(
                observations,
                "Strong day-of-week pattern",
                f"{highest_day_difference:.1f}% above average",
                f"{highest_day} generated "
                f"{highest_day_difference:.1f}% more revenue "
                f"than the average day.",
                "Review whether staffing, inventory, and operations "
                "should be adjusted for this day."
            )

        if lowest_day_difference <= -20:

            add_observation(
                observations,
                "Weak day-of-week pattern",
                f"{abs(lowest_day_difference):.1f}% below average",
                f"{lowest_day} generated "
                f"{abs(lowest_day_difference):.1f}% less revenue "
                f"than the average day.",
                "Investigate whether the lower revenue is consistent "
                "across locations and time periods."
            )

    # Location-specific peak periods

    location_peak_periods = {}

    for location in results["revenue_by_day_hour"]:

        location_hour_data = (
            results["revenue_by_day_hour"][location]
        )

        location_period_totals = {}

        for period in ["Morning", "Lunch", "Afternoon", "Evening"]:

            period_hours = {
                "Morning": range(8, 11),
                "Lunch": range(11, 14),
                "Afternoon": range(14, 17),
                "Evening": range(17, 21)
            }

            hours = period_hours[period]

            period_revenue = location_hour_data[
                [
                    hour
                    for hour in hours
                    if hour in location_hour_data.columns
                ]
            ].sum().sum()

            location_period_totals[period] = period_revenue

        location_peak_periods[location] = max(
            location_period_totals,
            key=location_period_totals.get
        )

    unique_peak_periods = set(
        location_peak_periods.values()
    )

    if len(unique_peak_periods) > 1:

        peak_summary = ", ".join(
            f"{location}: {period}"
            for location, period
            in location_peak_periods.items()
        )

        add_observation(
            observations,
            "Location-specific timing",
            "Different peak periods",
            f"Peak sales periods differ by location: "
            f"{peak_summary}.",
            "Review location-specific customer behavior "
            "and align staffing and inventory with local demand."
        )

    quarterly_observations[quarter] = observations

    #################################################
    # BEGINNING OF OBSERVATIONS SHEET
    #################################################

    observations_ws = wb.create_sheet(title="Observations")

    observations_ws.column_dimensions["A"].width = 28
    observations_ws.column_dimensions["B"].width = 20
    observations_ws.column_dimensions["C"].width = 65
    observations_ws.column_dimensions["D"].width = 65

    observations_ws["A1"] = "Key Observations"
    observations_ws["A1"].font = header_18
    observations_ws.row_dimensions[1].height = 60
    observations_ws.merge_cells("A1:D1")
    observations_ws["A1"].alignment = center_align

    observations_ws["A3"] = (
        "Automated findings highlighting significant trends "
        "and patterns with identified areas for investigation."
    )
    observations_ws["A3"].font = header_14
    observations_ws.row_dimensions[3].height = 36
    observations_ws.merge_cells("A3:D3")
    observations_ws["A3"].alignment = center_align

    # blank row
    observations_ws["A4"] = ""
    observations_ws.row_dimensions[4].height = 30

    # Table headers

    headers = [
        "Observation",
        "Metric",
        "Finding",
        "Suggested Investigation"
    ]

    for col_num, header in enumerate(headers, start=1):

        cell = observations_ws.cell(
            row=5,
            column=col_num,
            value=header
        )

        cell.font = header_14
        cell.alignment = center_align

    # Write observations
    
    observation_row = 6

    for observation in observations:

        observations_ws.cell(
            row=observation_row,
            column=1,
            value=observation["Observation"]
        )

        observations_ws.cell(
            row=observation_row,
            column=2,
            value=observation["Metric"]
        )

        observations_ws.cell(
            row=observation_row,
            column=3,
            value=observation["Finding"]
        )

        observations_ws.cell(
            row=observation_row,
            column=4,
            value=observation["Suggested Investigation"]
        )

        for col_num in range(1, 5):

            observations_ws.cell(
                row=observation_row,
                column=col_num
            ).alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

        observations_ws.row_dimensions[
            observation_row
        ].height = 45

        observation_row += 1    

    #################################################
    # END OF OBSERVATIONS SHEET
    #################################################

    # back to first sheet
    wb.active = kpi_sheet

    # Save the file
    report_name = (
    f"Performance Report - "
    f"{quarter} - {start_month} – {end_month} {year}.xlsx"
    )
    #------------------------------------
    print(f"Saving report: {report_name}")
    wb.save(report_name)


Saving report: Performance Report - 2023Q1 - April – June 2023.xlsx
Saving report: Performance Report - 2023Q2 - April – June 2023.xlsx
